# Reactive machine agents with Mesa (v3.x)

In this notebook you will:
- install **Mesa 3.x** with visualisation support,
- implement a **reactive agent** that monitors the temperature of a machine,
- build a simple **factory model** with several machine agents,
- start an **interactive visualisation** directly in the notebook and explore different parameters


## 1. Installation
First we install Mesa **3.0.3** including visualisation support.


In [ ]:
#!pip install -U mesa[viz]==3.0.3 altair==5.2.0 networkx solara

## 2. Mesa basics
Mesa models consist of three main parts:
- a **Model** class that manages the global state,
- one or more **Agent** classes that define the behaviour of individual agents,
- an optional **visualisation** to observe the model while it runs.

**Important changes in Mesa 3.x compared with v2.x:**
- `Agent.__init__` no longer needs `unique_id` – it is assigned automatically.
- The scheduler (`RandomActivation`) no longer exists → use `self.agents.shuffle_do("step")` instead.
- `model.schedule.agents` → `model.agents`.
- Removing agents: `self.remove()` instead of `self.model.schedule.remove(self)`.


## 3. Implementing the `MachineAgent`
The `MachineAgent` represents a single machine in a factory.


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Reactive agent that represents a machine and monitors its temperature.

    Rule:
        temperature > threshold            -> state = 'HOT'
        temperature > 0.75 * threshold     -> state = 'WARM'
        temperature < 20                   -> state = 'COOL'
        otherwise                          -> state = 'OK'
    """

    def __init__(self, model, threshold=70):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False

    def sense_temperature(self):
        """Simple sensor model: temperature + random noise."""
        if self.state != "HOT":
            noise = random.uniform(-1, 2)
            self.temperature += noise

    def decide(self):
        """Reactive decision rule based only on the current temperature."""
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        """Cool the machine down when it is overheated."""
        if self.state == "HOT":
            self.temperature -= 10

    def step(self):
        """Agent step = sense → decide → act."""
        self.sense_temperature()
        self.decide()
        self.act()


## 4. Implementing the `FactoryModel`
The model places several machines on a grid and activates them in random order. A **DataCollector** tracks how many machines are in the `"HOT"` and `"WARM"` states.


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")


def count_warm_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "WARM")


# Reference to the model that is currently displayed (used by post_process)
CURRENT_MODEL = {}


class FactoryModel(Model):
    """Simple factory model with a grid of machine agents."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70, seed=None):
        super().__init__(seed=seed)
        CURRENT_MODEL["model"] = self  # register the model that is currently displayed
        self.width = width
        self.height = height
        self.density = density
        self.threshold = threshold

        self.grid = MultiGrid(width, height, torus=False)

        for x in range(self.width):
            for y in range(self.height):
                if self.random.random() < self.density:
                    agent = MachineAgent(self, threshold=self.threshold)
                    self.grid.place_agent(agent, (x, y))

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
                "WarmMachines": count_warm_machines,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


## 5. Visualisation with Mesa 3.x (SolaraViz)
Mesa 3.x uses **SolaraViz** as its visualisation framework.

The `post_process` function is called after the agents have been drawn and allows arbitrary Matplotlib customisation. Important: it must be defined as a **closure** so that it always accesses the *current* model object – not a hard-wired global variable.

⚠️ **Note:** If the display does not work directly in the notebook,
save the code to a `.py` file and start it with `solara run file.py`.


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component


# ── Agent portrayal ─────────────────────────────────────────────────────────
def agent_portrayal(agent):
    return {
        "color": {
            "HOT":  "red",
            "WARM": "orange",
            "COOL": "skyblue",
            "OK":   "green",
        }.get(agent.state, "gray"),
        "size":   600,
        "marker": "s",
        "alpha":  0.9,
    }


# ── post_process: always draw the CURRENT model ────────────────────────────────
# SolaraViz creates a NEW model object on Reset and when parameters change.
# The model registers itself in CURRENT_MODEL in __init__, so that
# post_process always labels the agents of the model that is displayed
# (not those of the first instance).
def make_post_process(model):
    def post_process(ax):
        ax.figure.set_size_inches(6, 6)

        legend = ax.get_legend()
        if legend:
            legend.remove()

        for agent in CURRENT_MODEL.get("model", model).agents:
            if not isinstance(agent, MachineAgent):
                continue
            x, y = agent.pos
            ax.text(
                x, y,
                f"{agent.temperature:.1f}°C",
                ha="center", va="center",
                fontsize=7, color="white", fontweight="bold",
            )

        ax.grid(True, alpha=0.3)
    return post_process


# ── Components ────────────────────────────────────────────────────────────────
def make_components(model):
    """Create space_component and plot_component for the given model."""
    space = make_space_component(
        agent_portrayal,
        post_process=make_post_process(model),
    )
    plot = make_plot_component(
        {"HotMachines": "red", "WarmMachines": "orange"}
    )
    return space, plot


# ── Model parameters ────────────────────────────────────────────────────────────
model_params = {
    "width":  10,
    "height": 10,
    "density": {
        "type":  "SliderFloat",
        "value": 0.3,
        "label": "Density",
        "min":   0.1,
        "max":   1.0,
        "step":  0.1,
    },
    "threshold": {
        "type":  "SliderInt",
        "value": 70,
        "label": "Temperature threshold",
        "min":   30,
        "max":   100,
        "step":  5,
    },
}

# ── Model instance and visualisation ───────────────────────────────────────────
model_instance = FactoryModel(width=10, height=10, density=0.3, threshold=70)
space_component, plot_component = make_components(model_instance)

page = SolaraViz(
    model_instance,
    components=[space_component, plot_component],
    model_params=model_params,
    name="Reactive machine agents",
)
page


## 6. Testing the simulation without visualisation
If you only want to check the model logic, you can also run the model directly for a few steps.


In [ ]:
model = FactoryModel(width=10, height=10, density=0.3, threshold=70, seed=42)
for _ in range(20):
    model.step()

model.datacollector.get_model_vars_dataframe().tail()


## 7. Exploration tasks (for the lab)
Use the running model to explore the behaviour of the reactive agents:

1. **Change the threshold**
   - Start with `threshold=70` and then try 60 or 80.
   - What happens to the number of `"HOT"` machines over time?

2. **Change the density**
   - Increase or decrease the `density` parameter.
   - How does this influence the overall state of the system?

3. **Extend the state logic (optional)**
   - Add further states, e.g. `"CRITICAL"`, and map them to new colours in `agent_portrayal`.
   - Define your own temperature ranges for each state.

4. **(Advanced) Add maintenance agents**
   - Create a second agent type that moves through the grid and "repairs" overheated
     machines by resetting their temperature and state.

5. **(Advanced) Add order agents**
   - Create a third agent type that moves orders through the grid and
     looks for free machines by itself.


## 8. Sample solutions for the advanced tasks
We extend the model with two further agent types:

1. **`MaintenanceAgent`** – moves through the grid and repairs overheated machines.
2. **`OrderAgent`** – represents a production order, looks for a free machine
   and removes itself from the model with `self.remove()` (Mesa 3.x).


In [ ]:
from mesa import Agent
import random


class MachineAgent(Agent):
    """Extended reactive machine agent (identical to above)."""

    def __init__(self, model, threshold=70):
        super().__init__(model)
        self.temperature = 20.0
        self.threshold = threshold
        self.state = "OK"
        self.busy = False

    def sense_temperature(self):
        if self.state != "HOT":
            self.temperature += random.uniform(-1, 2)

    def decide(self):
        if self.temperature > self.threshold:
            self.state = "HOT"
        elif self.temperature > 0.75 * self.threshold:
            self.state = "WARM"
        elif self.temperature < 20:
            self.state = "COOL"
        else:
            self.state = "OK"

    def act(self):
        pass  # cooling is done by the MaintenanceAgent

    def step(self):
        self.sense_temperature()
        self.decide()
        self.act()


class MaintenanceAgent(Agent):
    """Moves through the grid and repairs HOT machines."""

    def step(self):
        neighbours = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=True
        )
        self.model.grid.move_agent(self, random.choice(neighbours))

        for agent in self.model.grid.get_cell_list_contents([self.pos]):
            if isinstance(agent, MachineAgent) and agent.state == "HOT":
                agent.temperature = agent.threshold - 40
                agent.state = "OK"
                agent.busy = False

class OrderAgent(Agent):
    """Production order: looks for a free machine and occupies it for 'duration' steps."""

    def __init__(self, model, duration=10):
        super().__init__(model)
        self.duration = duration
        self.assigned_machine = None   # ← reference to the occupied machine

    def step(self):
        # ── Case 1: no machine found yet → search ─────────────────────
        if self.assigned_machine is None:
            neighbours = self.model.grid.get_neighborhood(
                self.pos, moore=True, include_center=True
            )
            self.model.grid.move_agent(self, random.choice(neighbours))

            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT" and not agent.busy:
                    agent.busy = True
                    agent.temperature += 5
                    self.assigned_machine = agent   # ← remember the reference
                    break

        # ── Case 2: machine occupied → count down the duration ────────────────────
        else:
            self.duration -= 1
            for agent in self.model.grid.get_cell_list_contents([self.pos]):
                if isinstance(agent, MachineAgent) and agent.state != "HOT":
                    agent.temperature += 5 
            if self.duration <= 0:
                self.assigned_machine.busy = False
                self.model.grid.remove_agent(self)  # ← remove from the grid first
                self.remove()


In [ ]:
from mesa import Model
from mesa.space import MultiGrid
from mesa.datacollection import DataCollector


def count_hot_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "HOT")

def count_warm_machines(model):
    return sum(1 for a in model.agents if isinstance(a, MachineAgent) and a.state == "WARM")

def count_orders(model):
    return sum(1 for a in model.agents if isinstance(a, OrderAgent))


class FactoryModelExtended(Model):
    """Extended factory model with machine, maintenance and order agents."""

    def __init__(self, width=10, height=10, density=0.3, threshold=70,
                 n_maintenance=2, n_orders=5, seed=None):
        super().__init__(seed=seed)
        CURRENT_MODEL["model"] = self  # register the model that is currently displayed
        self.width = width
        self.height = height
        self.grid = MultiGrid(width, height, torus=False)

        for _ in range(n_orders):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(OrderAgent(self), pos)

        for x in range(width):
            for y in range(height):
                if self.random.random() < density:
                    self.grid.place_agent(MachineAgent(self, threshold=threshold), (x, y))

        for _ in range(n_maintenance):
            pos = (self.random.randrange(width), self.random.randrange(height))
            self.grid.place_agent(MaintenanceAgent(self), pos)

        

        self.datacollector = DataCollector(
            model_reporters={
                "HotMachines":  count_hot_machines,
                "WarmMachines": count_warm_machines,
                "Orders":    count_orders,
            }
        )
        self.datacollector.collect(self)

    def step(self):
        self.agents.shuffle_do("step")
        self.datacollector.collect(self)


### Visualisation of the extended model
Different agent types are drawn with different shapes and colours:
- **Machines** – coloured squares with temperature display
- **Maintenance agents** – turquoise circles with “W”
- **Order agents** – small black squares with order ID


In [ ]:
from mesa.visualization import SolaraViz, make_plot_component, make_space_component

# Set model parameters
DENSITY = 0.2 
THRESHOLD = 70 
N_MAINTENANCE = 2 
N_ORDERS = 30


# ── Agent portrayal ─────────────────────────────────────────────────────────
def agent_portrayal_ext(agent):
    if isinstance(agent, MachineAgent):
        return {
            "color": {
                "HOT":  "red",
                "WARM": "orange",
                "COOL": "skyblue",
                "OK":   "green",
            }.get(agent.state, "gray"),
            "size": 600, "marker": "s", "alpha": 0.7,
        }
    if isinstance(agent, MaintenanceAgent):
        return {"color": "cyan",  "size": 300, "marker": "o", "alpha": 0.95}
    if isinstance(agent, OrderAgent): 
        return {"color": "black", "size": 200, "marker": "s", "alpha": 0.85}
    return {}


# ── post_process: always draw the CURRENT model ────────────────────────────────
# SolaraViz creates a NEW model object on Reset and when parameters change.
# The model registers itself in CURRENT_MODEL in __init__, so that
# post_process always labels the agents of the model that is displayed
# (not those of the first instance).
def make_post_process_ext(model):
    def post_process(ax):
        ax.figure.set_size_inches(6, 6)
        legend = ax.get_legend()
        if legend:
            legend.remove()

        for agent in CURRENT_MODEL.get("model", model).agents:
            x, y = agent.pos
            if isinstance(agent, MachineAgent):
                ax.text(x, y, f"{agent.temperature:.1f}°C",
                        ha="center", va="center",
                        fontsize=6, color="white", fontweight="bold")
            elif isinstance(agent, MaintenanceAgent):
                ax.text(x, y, "W",
                        ha="center", va="center",
                        fontsize=9, color="black", fontweight="bold")
            elif isinstance(agent, OrderAgent):
                ax.text(x, y, str(agent.unique_id),
                        ha="center", va="center",
                        fontsize=6, color="white")

        ax.grid(True, alpha=0.3)
    return post_process


''' ── Model parameters ────────────────────────────────────────────────────────────
model_params_ext = {
    "width": 10, "height": 10,
    "density": {
        "type": "SliderFloat", "value": DENSITY,
        "label": "Machine density", "min": 0.1, "max": 1.0, "step": 0.1,
    },
    "threshold": {
        "type": "SliderInt", "value": THRESHOLD,
        "label": "Temperature threshold", "min": 30, "max": 100, "step": 5,
    },
    "n_maintenance": {
        "type": "SliderInt", "value": N_MAINTENANCE,
        "label": "Maintenance agents", "min": 0, "max": 10, "step": 1,
    },
    "n_orders": {
        "type": "SliderInt", "value": N_ORDERS,
        "label": "Order agents", "min": 10, "max": 100, "step": 1,
    },
}
'''

# ── Model instance and visualisation ───────────────────────────────────────────
model_ext = FactoryModelExtended(
    width=10, height=10, density=DENSITY, threshold=THRESHOLD, n_maintenance=N_MAINTENANCE, n_orders=N_ORDERS
)

space_ext = make_space_component(
    agent_portrayal_ext,
    post_process=make_post_process_ext(model_ext),
)
plot_ext = make_plot_component(
    {"HotMachines": "red", "WarmMachines": "orange", "Orders": "black"}
)

page_ext = SolaraViz(
    model_ext,
    components=[space_ext, plot_ext],
   # model_params=model_params_ext,
    name="Extended factory model",
    play_interval=200,   # ← 200 ms between steps = 5 fps
)
page_ext
